# C-13 Subgroup Fairness Metrics

**Control:** C-13 Subgroup Fairness Metrics, MAK-AI Responsible AI Toolkit.

**Learning goal:** compute group fairness numbers for a classifier (selection rate, demographic parity, equalised odds, equal opportunity, and calibration within each group), see why they can disagree with each other, and choose the ones that fit your use.

**Time to run:** about 3 minutes on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** UCI Adult (Becker and Kohavi, 1996), DOI 10.24432/C5XW20, licence CC BY 4.0, 48,842 rows, from https://archive.ics.uci.edu/dataset/2/adult. It is credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** Adult is a 1994 United States census extract. Its group labels (two sexes, five race categories) are US census categories of that year. The numbers below teach a method. They say nothing about the people your system serves, and the "Adapt here" cell shows how to choose groups that matter where you work.

## Linked risk areas and questions

Generated from content 5.2.0. Do not edit by hand.

Risk areas: 4

- IP-01: Algorithmic Discrimination
- IP-02: Aggregation Bias
- IP-04: Metric Selection Bias
- IP-05: Model Robustness Failure

Questions: 16: Q-PP-23, Q-IP-01, Q-IP-02, Q-IP-03, Q-IP-04, Q-IP-06, Q-IP-08, Q-IP-16, Q-IP-17, Q-IP-18, Q-IP-13, Q-IP-14, Q-IP-15, Q-IP-19, Q-IP-22, Q-IP-24

## When to use this notebook

Use it when you are answering a question about whether a model treats groups differently and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-IP-06`: which fairness metrics have you computed? (Step 6 below says which option each number here matches.)
- `Q-IP-04`: have you calculated performance metrics separately for each relevant subgroup?
- `Q-IP-14`: does aggregating data across populations hide differences between subgroups?
- `Q-IP-19`: do the model's errors have a different impact on different groups?

You need a table with one row per person or case, the true outcome, and either a model's predicted probabilities or the columns to train a simple model. If you have no model yet, the notebook trains a plain one so that you can practise on it.

In [ ]:
import hashlib
import importlib.metadata
import io
import os
import platform
import tempfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fairlearn.metrics import (
    MetricFrame,
    count,
    demographic_parity_difference,
    demographic_parity_ratio,
    equal_opportunity_difference,
    equalized_odds_difference,
    false_negative_rate,
    false_positive_rate,
    selection_rate,
    true_positive_rate,
)
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 20261030  # the one random seed; every random step below uses it
rng = np.random.default_rng(SEED)
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scikit-learn', 'fairlearn', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/2/data.csv'
DATA_SHA256 = '26b38f8623766396e2ba0d1a235a283db8ed993cc32bf32fc8aab09708c3bdf2'
DATA_FILE = 'uci-adult-2-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024  # a dataset file is never larger than this; a longer body is refused


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. The body is read in 1 MiB pieces and refused past MAX_DOWNLOAD_BYTES.
    `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    if Path(file_name).name != file_name:
        raise ValueError(f'the cache file name must be a plain name: {file_name}')

    def read_checked(stream):
        digest, pieces, size = hashlib.sha256(), [], 0
        for piece in iter(lambda: stream.read(1 << 20), b''):
            size += len(piece)
            if size > MAX_DOWNLOAD_BYTES:
                raise RuntimeError(f'{file_name}: the file is larger than {MAX_DOWNLOAD_BYTES:,} bytes; refused')
            digest.update(piece)
            pieces.append(piece)
        return b''.join(pieces), digest.hexdigest()

    path = CACHE_DIR / file_name
    if path.exists():
        with path.open('rb') as stream:
            data, found = read_checked(stream)
        source, remedy = f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data, found = read_checked(response)
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix='.part', delete=False)
        try:
            with partial:
                partial.write(data)
            os.replace(partial.name, path)  # the cache never holds a half-written file, whoever else is writing
        except BaseException:
            Path(partial.name).unlink(missing_ok=True)
            raise
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))
# Adult only (delete for your own data): the census test file writes the label with a trailing dot.
df['income'] = df['income'].str.rstrip('.')
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')
df.head(3)

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
LABEL = 'income'                 # the column that holds the true outcome
FAVOURABLE = '>50K'              # the value of LABEL that counts as the good outcome for a person
# Choosing groups: start from who the system's decisions reach where you work, not from the columns this example
# has. Ask the people affected, and those who speak for them, which differences have led to unequal treatment there
# (for example language, region or district, rural or urban, disability, age band, displaced or refugee status).
# Use those columns if your file has them, and add their combination when a harm may fall on the overlap. Write down
# any group that matters but is not in your data. Collect a new sensitive attribute only with the people's consent.
GROUP_COLUMNS = ['sex', 'race']  # the columns that define the groups you compare
NUMERIC = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
CATEGORICAL = ['workclass', 'marital-status', 'occupation', 'relationship']
SCORE_COLUMN = None              # name a column of your own model's scores (0 to 1) to skip the practice model
PRACTICE_MODEL = 'logistic'      # the practice model when SCORE_COLUMN is None: logistic, or boosted (gradient-boosted trees)
THRESHOLD = 0.5                  # a score at or above this is treated as the favourable prediction
MIN_GROUP_SIZE = 100             # groups with fewer people than this in the test set are set aside, not compared
N_BOOTSTRAP = 200                # how many resamples to draw for the intervals

FEATURES = NUMERIC + CATEGORICAL  # the model never sees GROUP_COLUMNS: see Step 1 for why that is not enough

### Step 1: a baseline model

**What this does.** It splits the data into a training part (70 percent) and a test part (30 percent) and fits a plain logistic regression that predicts the favourable outcome (set PRACTICE_MODEL to 'boosted' in the "Adapt here" cell to fit gradient-boosted trees, scikit-learn's HistGradientBoostingClassifier, instead). Every fairness number below is computed on the test part only, so the model has not seen those people. The group columns are left out of the model's inputs.

**What it shows.** A realistic stand-in for "the model the team has built". If you named `SCORE_COLUMN` above, your own scores are used and nothing is trained.

**What it does not show.** Leaving the group columns out is often called "fairness through unawareness", and it does not remove group differences. Other columns such as `relationship` or `occupation` can carry the same information. Whether a model with these inputs is acceptable is a judgment, not something this step can settle.

In [ ]:
y = (df[LABEL] == FAVOURABLE).astype(int)
train_index, test_index = train_test_split(df.index, test_size=0.3, stratify=y, random_state=SEED)

if SCORE_COLUMN is None:
    preprocess = ColumnTransformer([
        ('numbers', StandardScaler(), NUMERIC),
        ('categories', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ], sparse_threshold=0)  # dense output, which the boosted model needs
    if PRACTICE_MODEL == 'boosted':
        estimator = HistGradientBoostingClassifier(random_state=SEED)
    else:
        estimator = LogisticRegression(max_iter=1000)
    model = make_pipeline(preprocess, estimator)
    model.fit(df.loc[train_index, FEATURES], y[train_index])
    score = pd.Series(model.predict_proba(df[FEATURES])[:, 1], index=df.index)
else:
    score = df[SCORE_COLUMN].astype(float)

test = pd.DataFrame({'y': y[test_index], 'score': score[test_index]})
test['pred'] = (test['score'] >= THRESHOLD).astype(int)

groups = df.loc[test_index, GROUP_COLUMNS].astype(str)
groups['intersection'] = groups[GROUP_COLUMNS].agg(' / '.join, axis=1)  # every combination of the group columns
GROUPINGS = {name: groups[name] for name in GROUP_COLUMNS + ['intersection']}

print(f'test people: {len(test):,}')
print(f'overall accuracy: {accuracy_score(test.y, test.pred):.3f}')
print(f'overall selection rate (share predicted favourable): {test.pred.mean():.3f}')
print(f'share who truly have the favourable outcome: {test.y.mean():.3f}')

### Step 2: the numbers for each group

**What this does.** `MetricFrame` from Fairlearn takes the true outcomes, the predictions and a group column, and computes the same metrics separately for every group. Each metric answers one plain question:

- **count**: how many people are in the group. Small groups give shaky numbers.
- **selection rate**: out of everyone in the group, what share did the model label favourable?
- **true positive rate**: out of the people who really have the favourable outcome, what share did the model label favourable? Low means deserving people are wrongly denied.
- **false positive rate**: out of the people who do not have the favourable outcome, what share did the model label favourable? High means people are wrongly granted.
- **false negative rate**: the opposite of the true positive rate, the share of deserving people wrongly denied.
- **precision**: out of the people labelled favourable, what share really are?
- **accuracy**: out of everyone, what share did the model get right?

The same table is drawn for each group column and for their intersection (for example, women of one race), because a model can look fair on each column alone and still treat an intersection badly.

**What it shows.** Where the rates differ and by how much, group by group.

**What it does not show.** Whether a difference is real or just chance (Step 5 adds intervals), and whether it is unfair. Rates differ for many reasons, some of them in the world and not in the model.

In [ ]:
def precision_or_zero(y_true, y_pred):
    return precision_score(y_true, y_pred, zero_division=0)


METRICS = {
    'count': count,
    'selection rate': selection_rate,
    'true positive rate': true_positive_rate,
    'false positive rate': false_positive_rate,
    'false negative rate': false_negative_rate,
    'precision': precision_or_zero,
    'accuracy': accuracy_score,
}

frames = {name: MetricFrame(metrics=METRICS, y_true=test.y, y_pred=test.pred, sensitive_features=feature)
          for name, feature in GROUPINGS.items()}

for name, frame in frames.items():
    table = frame.by_group.copy()
    table['set aside (too small)'] = table['count'] < MIN_GROUP_SIZE
    print(f'--- by {name} ---')
    print(table.round(3).to_string())
    print()

### Step 3: one number per metric for the whole comparison

**What this does.** It condenses the group table into the single gap numbers that fairness papers and tools quote. Groups smaller than `MIN_GROUP_SIZE` are set aside first, because a group of ten people can produce an extreme rate by chance and then it alone decides the gap.

- **Demographic parity difference**: the largest gap in selection rate between any two groups. Zero means every group is labelled favourable at the same rate. The ratio version divides the lowest rate by the highest, so 1 means equal and 0.8 is the "four-fifths" rule of thumb from US employment practice.
- **Equal opportunity difference**: the largest gap in the true positive rate. Zero means deserving people have the same chance of being recognised in every group.
- **Equalised odds difference**: the larger of the true-positive-rate gap and the false-positive-rate gap. Zero means both kinds of error are shared evenly.

**What it shows.** How far apart the groups are on each definition of fair.

**What it does not show.** Which definition is the right one. They measure different things, and when the groups have different real rates of the favourable outcome they cannot all be zero at once: equal selection rates and equal error rates can then hold together only for a model whose predictions carry no information about the true outcome, such as one that labels everyone the same. Picking one is the "Stop and decide" step.

It also does not make every gap trustworthy. The size rule counts everyone in a group, but the true positive rate uses only the group's people who really have the favourable outcome, and the false positive rate only those who do not. A group can meet the size rule and still hold only a handful of people who really have the favourable outcome, and then its true positive rate, and any gap it sets, can swing by chance. Read the true-positive-rate gaps beside their intervals in Step 5.

In [ ]:
gap_rows = []
for name, feature in GROUPINGS.items():
    sizes = feature.map(feature.value_counts())
    keep = sizes >= MIN_GROUP_SIZE
    kept = dict(y_true=test.y[keep], y_pred=test.pred[keep], sensitive_features=feature[keep])
    gap_rows.append({
        'grouping': name,
        'groups compared': feature[keep].nunique(),
        'groups set aside': feature.nunique() - feature[keep].nunique(),
        'demographic parity difference': demographic_parity_difference(**kept),
        'demographic parity ratio': demographic_parity_ratio(**kept),
        'equal opportunity difference': equal_opportunity_difference(**kept),
        'equalized odds difference': equalized_odds_difference(**kept),
    })
gaps = pd.DataFrame(gap_rows).set_index('grouping')
gaps.round(3)

### Step 4: calibration within each group

**What this does.** A model that outputs a score should mean the same thing for everyone: among people given a score near 0.7, about 70 percent should really have the favourable outcome, whatever their group. A calibration curve groups people by score and plots the real rate against the score. A group whose curve sits below the diagonal is being over-scored, and a group above is under-scored. The Brier score is the average squared gap between each person's score and their real outcome (0 or 1), one number per group, and lower is better. It is not a calibration number on its own: it also changes with how well the score separates the two outcomes and with how common the favourable outcome is in the group, so a group where the favourable outcome is rare tends to get a lower Brier score even when its curve is no closer to the diagonal. Compare groups on the curves, and read the Brier score only beside them.

**What it shows.** Whether a score is equally trustworthy across groups. This is a different fairness idea from Steps 2 and 3, which look at decisions after a threshold.

**What it does not show.** A model can be well calibrated in every group and still have unequal selection or error rates, so good calibration does not settle the other questions. Curves for small groups wobble a lot; only groups of at least `MIN_GROUP_SIZE` are drawn.

In [ ]:
calibration_rows = []
fig, axes = plt.subplots(1, len(GROUP_COLUMNS), figsize=(5.5 * len(GROUP_COLUMNS), 4.5), squeeze=False)
for axis, name in zip(axes[0], GROUP_COLUMNS):
    feature = GROUPINGS[name]
    axis.plot([0, 1], [0, 1], linestyle='--', color='grey', label='perfectly calibrated')
    for label, members in test.groupby(feature):
        if len(members) < MIN_GROUP_SIZE:
            continue
        bins = 10 if len(members) >= 1000 else 5
        real, predicted = calibration_curve(members.y, members.score, n_bins=bins, strategy='quantile')
        axis.plot(predicted, real, marker='o', label=f'{label} (n={len(members):,})')
        calibration_rows.append({'grouping': name, 'group': label, 'n': len(members),
                                 'brier score': brier_score_loss(members.y, members.score)})
    axis.set_xlabel('score given by the model')
    axis.set_ylabel('share who really have the favourable outcome')
    axis.set_title(f'Calibration by {name}')
    axis.legend(fontsize=8)
fig.tight_layout()
calibration = pd.DataFrame(calibration_rows)
calibration.round(3)

### Step 5: how sure can we be? Group sizes and bootstrap intervals

**What this does.** The test set is a sample. If we had drawn a slightly different sample, the gaps would be slightly different. The bootstrap imitates that: it draws `N_BOOTSTRAP` new test sets of the same size by picking people at random with replacement, recomputes the selection-rate gap and the true-positive-rate gap each time, and reports the range that holds the middle 95 percent of the results.

**What it shows.** How much of a gap could be luck. Each gap here is the highest group value minus the lowest, so it is never below zero, and chance alone pushes it above zero, more so when groups are small or there are many of them. So an interval will not reach below zero; one whose low end is close to zero means the data cannot tell a real gap from noise. A narrow interval well above zero means the gap is likely to show up again on similar data.

**What it does not show.** The bootstrap does not fix a test set that does not look like your real population (the data itself may be unrepresentative), and it does not make a tiny group trustworthy: the groups set aside in Step 3 stay set aside.

In [ ]:
def gap_between_groups(y_true, y_pred, group):
    """Largest difference between groups in selection rate and in true positive rate."""
    return MetricFrame(metrics={'selection rate': selection_rate, 'true positive rate': true_positive_rate},
                       y_true=y_true, y_pred=y_pred, sensitive_features=group).difference()


interval_rows = []
for name, feature in GROUPINGS.items():
    keep = (feature.map(feature.value_counts()) >= MIN_GROUP_SIZE).to_numpy()
    y_kept, pred_kept, group_kept = test.y.to_numpy()[keep], test.pred.to_numpy()[keep], feature.to_numpy()[keep]
    observed = gap_between_groups(y_kept, pred_kept, group_kept)
    draws = []
    for _ in range(N_BOOTSTRAP):
        pick = rng.integers(0, len(y_kept), len(y_kept))
        draws.append(gap_between_groups(y_kept[pick], pred_kept[pick], group_kept[pick]))
    draws = pd.DataFrame(draws)
    for metric in draws.columns:
        low, high = draws[metric].quantile([0.025, 0.975])
        interval_rows.append({'grouping': name, 'gap in': metric, 'observed gap': observed[metric],
                              'interval low': low, 'interval high': high})
intervals = pd.DataFrame(interval_rows)
intervals.round(3)

### Step 6: which harm does each metric watch?

A classifier can be wrong in two ways, and the people who pay for each are different. **Wrongly denied** means someone who should get the favourable outcome (a loan, a place, a referral) does not. **Wrongly granted** means someone who should not get it does, which can harm the person (a loan they cannot repay) or the programme (resources spent where they are not needed). The table says which metric watches which harm, and which option of the platform's question `Q-IP-06` ("Which fairness metrics have you computed?") the number corresponds to.

| Number in this notebook | What it watches | Harm | Option of `Q-IP-06` |
|---|---|---|---|
| Demographic parity difference and ratio (selection rate) | Whether groups are labelled favourable at the same rate, ignoring the true outcome | Neither directly: it looks at who gets the outcome, not at whether they should | Statistical Parity |
| True positive rate, equal opportunity difference | Whether people who should get the outcome are recognised equally | Wrongly denied | False Negative Error Rate Balance |
| False positive rate | Whether people who should not get the outcome are granted it equally | Wrongly granted | False Positive Error Rate Balance |
| Equalised odds difference | Both of the two above together | Wrongly denied and wrongly granted | False Negative Error Rate Balance and False Positive Error Rate Balance |
| Precision | Whether a favourable label means the same in every group | Wrongly granted, seen from the side of the people labelled favourable | Predictive Parity |
| Accuracy | Whether the model is right equally often | Both, blended | Overall Accuracy Equality |
| Calibration curve (the Brier score beside it is a broader accuracy number, not a calibration measure) | Whether a score means the same in every group | Both, through the score | Test Fairness / Well-Calibration |

Not computed here: Conditional Statistical Parity, Treatment Equality and Conditional Use Accuracy Equality. Do not tick an option of `Q-IP-06` that this notebook did not compute.

### Stop and decide: which metric matters for this use, and why

The notebook cannot decide this for you, because it depends on what the decision does to people.

Questions to ask in the room, with the people who will be affected or who speak for them:

1. Who is hurt more by a wrong answer, a person wrongly denied or a person wrongly granted? Could the answer differ by group?
2. Is the favourable outcome a benefit people want (a place, a loan, a service) or a burden (a flag, an inspection)? The harm flips if it is a burden.
3. Do you believe the real rate of the favourable outcome differs across these groups, and do you trust the recorded outcome to show it? If the rates truly differ, equal selection rates mean people with the same true outcome are treated differently across groups. If the recorded outcome itself carries past unfairness, the error rates in Steps 2 and 3 are measured against that unfairness.
4. Which gaps in Steps 3 and 5 are big enough, and certain enough, that you would act on them? Who decides, and where is that written down?
5. Which groups are missing because they were too small to compare, and what will you do about them?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
CHOSEN_METRICS = []       # for example: ['true positive rate', 'false positive rate']
WHY_THESE_METRICS = ''    # why these fit this use, in your words
HARM_THAT_MATTERS_MORE = ''   # who is hurt more by a wrong answer
GAPS_WE_WILL_ACT_ON = ''  # which gaps are big and certain enough to act on, and who decides

In [ ]:
import zipfile

EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of numbers, one row per group. They hold no row of the data.
all_groups = pd.concat({name: frame.by_group for name, frame in frames.items()}, names=['grouping', 'group'])
all_groups.round(4).to_csv(EVIDENCE_DIR / 'C-13_metrics_by_group.csv')
gaps.round(4).to_csv(EVIDENCE_DIR / 'C-13_gaps.csv')
intervals.round(4).to_csv(EVIDENCE_DIR / 'C-13_gap_intervals.csv', index=False)
calibration.round(4).to_csv(EVIDENCE_DIR / 'C-13_calibration_by_group.csv', index=False)
fig.savefig(EVIDENCE_DIR / 'C-13_calibration.png', dpi=110, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
choice = CHOSEN_METRICS if CHOSEN_METRICS else 'not decided yet'
summary = f"""# C-13 Subgroup Fairness Metrics: evidence summary

Attach the file `C-13_evidence.zip`, which the notebook writes next to the `evidence` folder, to the decision it supports (questions Q-IP-04, Q-IP-06, Q-IP-14, Q-IP-19). It holds this notebook's own summary, tables and pictures, numbers only, never the data.

## What was run
- Groups compared: {', '.join(GROUP_COLUMNS)} and their intersection
- Outcome column: {LABEL}, favourable value {FAVOURABLE}
- Decision threshold: {THRESHOLD}
- Groups smaller than {MIN_GROUP_SIZE} people in the test set were set aside
- Test people: {len(test):,}, bootstrap resamples: {N_BOOTSTRAP}
- Model: {('practice model: ' + PRACTICE_MODEL) if SCORE_COLUMN is None else 'scores from the column ' + SCORE_COLUMN}

## Gap numbers (groups of enough size only)
{gaps.round(3).to_string()}

## Choices made in the stop-and-decide cell
- Metrics chosen: {choice}
- Why: {WHY_THESE_METRICS or 'not written yet'}
- Harm that matters more: {HARM_THAT_MATTERS_MORE or 'not written yet'}
- Gaps we will act on: {GAPS_WE_WILL_ACT_ON or 'not written yet'}

## Files beside this summary
C-13_metrics_by_group.csv, C-13_gaps.csv, C-13_gap_intervals.csv, C-13_calibration_by_group.csv, C-13_calibration.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-13_summary.md').write_text(summary, encoding='utf-8')
# One ZIP of the files this cell wrote, next to the folder, not inside it. Only the files this cell wrote enter it.
# It is built under a .part name and moved into place only when complete, so an error while it is built leaves no ZIP.
Path('C-13_evidence.zip').unlink(missing_ok=True)
Path('C-13_evidence.zip.part').unlink(missing_ok=True)
with zipfile.ZipFile('C-13_evidence.zip.part', 'x', zipfile.ZIP_DEFLATED) as archive:
    for name in [
        'C-13_calibration.png', 'C-13_calibration_by_group.csv', 'C-13_gap_intervals.csv',
        'C-13_gaps.csv', 'C-13_metrics_by_group.csv', 'C-13_summary.md'
    ]:
        path = EVIDENCE_DIR / name
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f'{name} is missing or is a link, so no ZIP was made. Run this notebook again from the top; if the file is a link, delete it first.')
        archive.write(path, name)
os.replace('C-13_evidence.zip.part', 'C-13_evidence.zip')
print(summary)

## What this does not show, and where to go next

- **It does not say your model is fair or unfair.** It measures gaps. Whether a gap is acceptable is a judgment for the people the system affects, the team and whoever is accountable, and the "Stop and decide" cell records it.
- **It covers the groups you gave it.** A group you did not name, or one too small to compare, is not examined. Groups that matter locally (language, region, disability, a displaced community) may not be columns in your file at all. Say so in the record.
- **Group fairness is not individual fairness.** Two similar people can still be treated differently while the group averages match.
- **A fairness metric is not a legal or ethical test.** It does not tell you what your country's law requires.
- **The data may hide problems.** A skewed or mislabelled dataset gives clean-looking numbers about a broken picture. See the notebooks for data quality (C-03) and bias in data (C-04).

Where to go next: C-14 Disaggregated Performance Analysis (performance by group with intervals, when you want to report it), C-15 Bias Mitigation Techniques (what to try when a gap is too large), and the model card template, whose disaggregated results table takes the numbers from this notebook's CSV file.